# Example: End-To-End *De Novo* Protein Design Pipeline

## Overview

This notebook demonstrates an end-to-end protein design workflow using three deep learning networks from the Institute for Protein Design:

| Step | Model | Purpose |
|------|-------|---------|
| 1. **Backbone Generation** | RFD3 | Generate novel protein backbones via diffusion |
| 2. **Sequence Design** | MPNN | Design amino acid sequences for the generated backbone |
| 3. **Structure Validation** | RF3 | Predict the structure from designed sequence to validate designability |

All models are unified through [AtomWorks](https://github.com/RosettaCommons/atomworks) (for both inference and training), relying on Biotite `AtomArray` objects.

This notebook assumes you have the base checkpoints downloaded: `foundry install rfd3 ligandmpnn rf3`. You can also specify the paths directly yourself if you wish. You can register your foundry venv to jupyter with: `python -m ipykernel install --user --name=foundry --display-name "foundry"`.

### Pipeline Flow
```
RFD3 (backbone) → MPNN (sequence) → RF3 (validation) → RMSD comparison
```
---

## Section 0: Installation

Install the Foundry package (includes RFD3, MPNN, and RF3):

```bash
pip install 'rc-foundry[all]'
```

Download the model weights (~6GB total, takes a couple minutes):

```bash
foundry install rfd3 ligandmpnn rf3
```

---

In [1]:
# Shared utilities for visualization (from AtomWorks)
from atomworks.io.utils.visualize import view
import warnings
warnings.filterwarnings('ignore', module='atomworks')

Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
Environment variable PDB_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository


## Section 1: Backbone Generation with RFD3

RFdiffusion3 (RFD3) generates *de novo* all-atom proteins that meet specific conditioning requirements.

**Parameters Used** *(many more are available for more complex protein design tasks)*:
- `length`: Target protein length in residues
- `diffusion_batch_size`: Number of structures to generate per batch
- `n_batches`: Number of batches to run

**Outputs:** Dictionary of `RFD3Output` objects.

In [2]:
from lightning.fabric import seed_everything
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine

# Set seed for reproducibility
seed_everything(0)

# Configure RFD3 inference
config = RFD3InferenceConfig(
    diffusion_batch_size=2,  # Generate 2 structures per batch
)

# Initialize engine and run generation
model = RFD3InferenceEngine(**config)
outputs = model.run(
    inputs='enzyme_design.json',      # None for unconditional generation
    out_dir=None,     # None to return in memory (no file output)
    n_batches=2,      # Generate 1 batch
)

12:31:23 DEBUG transforms: Debug mode is on
Seed set to 0
12:31:25 INFO rfd3.engine: [rank: 0] Prevalidating design specification for example: enzyme_design_M0255_1mg5_unfixed
12:31:25 WARNING atomworks.io: We can't fix formal charges without building from templates, as we need to know the true number of hydrogens bonded to a given atom, not the inferred number. This may lead to occasional inaccuracies after adding inter-residue bonds. To avoid this and fix formal charges, set `add_missing_atoms = True`.
12:31:25 WARNING atomworks.io: Chain A contains both polymer and non-polymer residues; separating them for processing, naming the non-polymer residues as B.
Using bfloat16 Automatic Mixed Precision (AMP)
12:31:34 WARNING atomworks.io: We can't fix formal charges without building from templates, as we need to know the true number of hydrogens bonded to a given atom, not the inferred number. This may lead to occasional inaccuracies after adding inter-residue bonds. To avoid this and fix 

In [3]:
# View generated example IDs (one key per generated structure)
outputs.keys()

dict_keys(['enzyme_design_M0255_1mg5_unfixed_0', 'enzyme_design_M0255_1mg5_unfixed_1'])

In [4]:
# Inspect RFD3 outputs and extract the generated backbone
for idx, data in outputs.items():
    print(f"Batch {idx}: {len(data)} structure(s)")
    print(f"  Output type: {type(data[0]).__name__}")
    print(f"  AtomArray: {data[0].atom_array}")

# Extract the first generated backbone for downstream use
first_key = next(iter(outputs.keys()))
atom_array = outputs[first_key][0].atom_array

# Visualize the generated backbone
view(atom_array)

Batch enzyme_design_M0255_1mg5_unfixed_0: 2 structure(s)
  Output type: RFD3Output
  AtomArray:     A       1  MET N      N        13.345   -2.697    7.173
    A       1  MET CA     C        13.408   -1.261    6.896
    A       1  MET C      C        12.064   -0.746    6.508
    A       1  MET O      O        11.343   -1.387    5.724
    A       1  MET CB     C        14.417   -0.981    5.781
    A       1  MET CG     C        14.571    0.491    5.454
    A       1  MET SD     S        15.780    0.827    4.172
    A       1  MET CE     C        14.958    0.235    2.733
    A       2  VAL N      N        11.713    0.423    6.983
    A       2  VAL CA     C        10.416    1.044    6.638
    A       2  VAL C      C        10.667    2.156    5.651
    A       2  VAL O      O        11.471    3.047    5.895
    A       2  VAL CB     C         9.719    1.590    7.884
    A       2  VAL CG1    C         8.469    2.270    7.514
    A       2  VAL CG2    C         9.474    0.462    8.858
    

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---

## Section 2: Sequence Design with MPNN

Protein and Ligand MPNN (Message Passing Neural Network) designs amino acid sequences that will fold into a target backbone structure.

**Model Options:**
- `protein_mpnn`: Original ProteinMPNN for protein-only design
- `ligand_mpnn`: Extended model supporting ligand-aware design

**Key Parameters:**
- `batch_size`: Number of sequences to generate per structure
- `remove_waters`: Whether to exclude water molecules from context

In [6]:
from collections import defaultdict
from mpnn.inference_engines.mpnn import MPNNInferenceEngine


# Configure MPNN inference engine
# See mpnn.utils.inference.MPNN_GLOBAL_INFERENCE_DEFAULTS for all options
engine_config = {
    "model_type": "ligand_mpnn",  # or "protein_mpnn" for vanilla ProteinMPNN
    "is_legacy_weights": True,    # Required for now for ligand_mpnn and protein_mpnn
    "out_directory": None,        # Return results in memory
    "write_structures": False,
    "write_fasta": False,
}

mpnn_config = defaultdict(list)
for idx, data in outputs.items():
    mpnn_config['input_dicts'].append({
        "name": f"{idx}",
        "batch_size": 10,
        "remove_waters": True,
    })
    mpnn_config['atom_arrays'].append(data[0].atom_array)

# Configure per-input inference options
# See mpnn.utils.inference.MPNN_PER_INPUT_INFERENCE_DEFAULTS for all options
input_configs = [
    {
        "batch_size": 10,         # Generate 10 sequences per structure
        "remove_waters": True,
    }
]

# Run sequence design on the RFD3-generated backbone
model = MPNNInferenceEngine(**engine_config)
mpnn_outputs = model.run(**mpnn_config)

In [7]:
from biotite.structure import get_residue_starts
from biotite.sequence import ProteinSequence

# Extract and display the designed sequences
print(f"Generated {len(mpnn_outputs)} designed sequences:\n")

for mpnn in mpnn_outputs:
    print(f"Sequence {mpnn.input_dict['name']}_{mpnn.output_dict['design_idx']}: {mpnn.output_dict['designed_sequence']}")

Generated 20 designed sequences:

Sequence enzyme_design_M0255_1mg5_unfixed_0_0: RVLACGGLSGRILELPLETQVAVARAVRDAGADGLAVFVGSGPDAAPLETAAREAPVLEAATGVPVWVVIIRPTPEATAAALDEAKALNPALRVGVGVNVSRDTTLEEAVARMRALLAAVGDRADAVFVNAGVDWDIALALARIALDAGKDVVIHAPQAGTPVHEAAAAGDVAGARAAAAMAVAARERILAA
Sequence enzyme_design_M0255_1mg5_unfixed_0_1: RILAVAGLGGTILDLPLETQVAVARAAAEAGADGLGVYYGSGPTACPLEAAAARLPQLEAATGVPVYAVVIRPTPEETAAAVDRAKALNPSLKVACGVNVSETTTLEEAVARMKALLAAVGDRADAIHVNAGVDEEIALALAKIALDAGKDVVVEAPQAGTPIARAANAGDVEGARAAARLAVAAKERLLAA
Sequence enzyme_design_M0255_1mg5_unfixed_0_2: RILACGGLGGTILDLPLETQVAVARACRDAGADGLSVFVGSGPTAAPLEKAAAAAPVLEAATGVPVYVTIIRPTPEATAAAVDRAKALNPSLKVAAGVNVSKDTTLEEAVARMKALLAAVGDRVDAVHVNAGVDPAIAEALAKLALDAGKDVVVEAPQAGTPVARAAGAGDVEGARAAAEMAVAAKRRLEAA
Sequence enzyme_design_M0255_1mg5_unfixed_0_3: MILACAGLSGRILDLPLEVQVAVARAARDAGADGLAVFVGSGPDAAPLEKAAAEAPVLEAATGVPVYVVIIRPTPEETLAAVREAKALNPALRVATGTNISKTTTLEQAVANMRALLEAVGDLADAVSVNAGVDREIALALAKIALDAGKDVVVHAPQAGTPVHIAAAAGDVAGARAAAEMAVAAKRELEAA
Sequen

---

## Section 3: Structure Prediction with RF3

RF3 (RoseTTAFold 3) predicts protein structures from sequences. By re-folding the MPNN-designed sequence, we can validate whether the design is likely to adopt the intended backbone structure.

**Outputs:** `RF3Output` objects containing:
- `atom_array`: Predicted structure as Biotite AtomArray
- `summary_confidences`: Overall confidence metrics (pLDDT, PAE, pTM, etc.)
- `confidences`: Per-atom/residue confidence scores

**Confidence Metrics:**
| Metric | Description |
|--------|-------------|
| pLDDT | Per-residue confidence (0-1, higher is better) |
| PAE | Predicted Aligned Error (lower is better) |
| pTM | Predicted TM-score |
| ranking_score | Overall model quality score |

In [8]:
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.utils.inference import InferenceInput


# Initialize RF3 inference engine
inference_engine = RF3InferenceEngine(ckpt_path='rf3', verbose=False)

# Create input from the MPNN-designed structure (first design)
# This re-folds the sequence to validate it adopts the intended structure
inputs_structure = [InferenceInput.from_atom_array(mpnn.atom_array, example_id=f"{mpnn.input_dict['name']}_{mpnn.output_dict['design_idx']}") for mpnn in mpnn_outputs]
rf3_outputs = inference_engine.run(inputs=inputs_structure)

# Outputs: dict mapping example_id -> list[RF3Output] (multiple models per input)
print(f"Output keys: {rf3_outputs.keys()}")
# print(f"Number of models for 'example_protein': {len(rf3_outputs['example_protein'])}")

13:08:10 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
13:08:10 WARNING atomworks.io: Adding missing atoms will erase extra fields. If you just want to load a structure with the given extra fields, you should probably use the much faster 'load_any' function from atomworks.io.utils.io_utils instead of 'parse'. Parse is meant for cleaning up structures from the RCSB PDB.
13:08:11 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
13:08:11 WARNING atomworks.io: Adding missing atoms will erase extra fields. If you just want to load a structure with the given extra fields, you should probably use the much faster 'load_any' function from atomworks.io.utils.io_utils instead of 'parse'. Parse is meant for cleaning up structures from the RCSB PDB.
13:08:11 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
13:08:11 WARNING atomworks.io: Adding missing at

Output keys: dict_keys(['enzyme_design_M0255_1mg5_unfixed_1_0', 'enzyme_design_M0255_1mg5_unfixed_1_1', 'enzyme_design_M0255_1mg5_unfixed_1_2', 'enzyme_design_M0255_1mg5_unfixed_1_3', 'enzyme_design_M0255_1mg5_unfixed_1_4', 'enzyme_design_M0255_1mg5_unfixed_1_5', 'enzyme_design_M0255_1mg5_unfixed_1_6', 'enzyme_design_M0255_1mg5_unfixed_1_7', 'enzyme_design_M0255_1mg5_unfixed_1_8', 'enzyme_design_M0255_1mg5_unfixed_1_9', 'enzyme_design_M0255_1mg5_unfixed_0_0', 'enzyme_design_M0255_1mg5_unfixed_0_1', 'enzyme_design_M0255_1mg5_unfixed_0_2', 'enzyme_design_M0255_1mg5_unfixed_0_3', 'enzyme_design_M0255_1mg5_unfixed_0_4', 'enzyme_design_M0255_1mg5_unfixed_0_5', 'enzyme_design_M0255_1mg5_unfixed_0_6', 'enzyme_design_M0255_1mg5_unfixed_0_7', 'enzyme_design_M0255_1mg5_unfixed_0_8', 'enzyme_design_M0255_1mg5_unfixed_0_9'])


In [10]:
# Extract the top-ranked prediction
rf3_output = rf3_outputs["enzyme_design_M0255_1mg5_unfixed_1_0"][0]

# Inspect RF3Output structure
print(f"RF3Output contains:")
print(f"  - atom_array: {len(rf3_output.atom_array)} atoms")
print(f"  - summary_confidences: {list(rf3_output.summary_confidences.keys())}")
print(f"  - confidences: {list(rf3_output.confidences.keys()) if rf3_output.confidences else None}")

# Visualize the predicted structure
view(rf3_output.atom_array)

RF3Output contains:
  - atom_array: 1441 atoms
  - summary_confidences: ['chain_ptm', 'chain_pair_pae_min', 'chain_pair_pde_min', 'chain_pair_pae', 'chain_pair_pde', 'overall_plddt', 'overall_pde', 'overall_pae', 'ptm', 'iptm', 'has_clash', 'ranking_score']
  - confidences: ['atom_chain_ids', 'atom_plddts', 'pae', 'token_chain_ids', 'token_res_ids']


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [11]:
# Summary confidences: overall model quality metrics
summary = rf3_output.summary_confidences

print("=== Summary Confidences ===")
print(f"  Overall pLDDT:    {summary['overall_plddt']:.3f}")
print(f"  Overall PAE:      {summary['overall_pae']:.2f} A")
print(f"  Overall PDE:      {summary['overall_pde']:.3f}")
print(f"  pTM:              {summary['ptm']:.3f}")
print(f"  ipTM:             {summary.get('iptm', 'N/A (single chain)')}")
print(f"  Ranking score:    {summary['ranking_score']:.3f}")
print(f"  Has clash:        {summary['has_clash']}")

=== Summary Confidences ===
  Overall pLDDT:    0.833
  Overall PAE:      5.15 A
  Overall PDE:      0.916
  pTM:              0.916
  ipTM:             0.8690127730369568
  Ranking score:    0.878
  Has clash:        False


In [12]:
# Detailed per-atom/residue confidences
conf = rf3_output.confidences

print("=== Per-Atom/Residue Confidences ===")
print(f"  atom_plddts:      {len(conf['atom_plddts'])} values (one per atom)")
print(f"  atom_chain_ids:   {len(conf['atom_chain_ids'])} values")
print(f"  token_chain_ids:  {len(conf['token_chain_ids'])} values (one per residue)")
print(f"  token_res_ids:    {len(conf['token_res_ids'])} values")
print(f"  PAE matrix:       {len(conf['pae'])}x{len(conf['pae'][0])}")

# Preview first 10 atom pLDDT scores
import numpy as np
print(f"\nFirst 10 atom pLDDTs: {np.round(conf['atom_plddts'][:10], 2).tolist()}")

=== Per-Atom/Residue Confidences ===
  atom_plddts:      1441 values (one per atom)
  atom_chain_ids:   1441 values
  token_chain_ids:  237 values (one per residue)
  token_res_ids:    237 values
  PAE matrix:       237x237

First 10 atom pLDDTs: [0.77, 0.78, 0.79, 0.75, 0.75, 0.82, 0.83, 0.84, 0.81, 0.81]


---

## Section 4: Validation and Export

The final step compares the RF3-predicted structure against the original RFD3-generated backbone. A low backbone RMSD indicates the designed sequence is likely to fold into the intended structure (high designability).

In [13]:
from biotite.structure import rmsd, superimpose
import biotite.structure as struc
import numpy as np

# print(len([mpnn_output.atom_array for mpnn_output in mpnn_outputs]))
# print(len([rf3_output[0] for rf3_output in rf3_outputs.values()]))
# [data[0].atom_array for _, data in outputs.items()]


for mpnn_output in mpnn_outputs:
    id = f"{mpnn_output.input_dict['name']}_{mpnn_output.output_dict['design_idx']}"
    print(id)
    aa_generated = mpnn_output.atom_array

    rf3_output = rf3_outputs[id]
    # Extract the top-ranked prediction
    aa_refolded = rf3_output[0].atom_array

    # Filter to backbone atoms (N, CA, C, O)
    # bb_generated = aa_generated[struc.filter_peptide_backbone(aa_generated)]
    # bb_refolded = aa_refolded[struc.filter_peptide_backbone(aa_refolded)]
    bb_generated = aa_generated[struc.filter_amino_acids(aa_generated) & np.isin(aa_generated.atom_name, ('N', 'CA', 'C', 'O'))]
    bb_refolded = aa_refolded[struc.filter_amino_acids(aa_refolded) & np.isin(aa_refolded.atom_name, ('N', 'CA', 'C', 'O'))]

    # Superimpose structures and calculate RMSD
    bb_refolded_fitted, transform = superimpose(bb_generated, bb_refolded)
    rmsd_value = rmsd(bb_generated, bb_refolded_fitted)

    
    print(f"Backbone RMSD: {rmsd_value:.2f} A {'Excellent' if rmsd_value < 1.0 else 'Good' if rmsd_value < 2.0 else 'Moderate'} designability\n")


enzyme_design_M0255_1mg5_unfixed_0_0
Backbone RMSD: 0.92 A Excellent designability

enzyme_design_M0255_1mg5_unfixed_0_1
Backbone RMSD: 0.83 A Excellent designability

enzyme_design_M0255_1mg5_unfixed_0_2
Backbone RMSD: 1.01 A Good designability

enzyme_design_M0255_1mg5_unfixed_0_3
Backbone RMSD: 0.92 A Excellent designability

enzyme_design_M0255_1mg5_unfixed_0_4
Backbone RMSD: 1.29 A Good designability

enzyme_design_M0255_1mg5_unfixed_0_5
Backbone RMSD: 1.27 A Good designability

enzyme_design_M0255_1mg5_unfixed_0_6
Backbone RMSD: 0.88 A Excellent designability

enzyme_design_M0255_1mg5_unfixed_0_7
Backbone RMSD: 0.80 A Excellent designability

enzyme_design_M0255_1mg5_unfixed_0_8
Backbone RMSD: 1.10 A Good designability

enzyme_design_M0255_1mg5_unfixed_0_9
Backbone RMSD: 0.87 A Excellent designability

enzyme_design_M0255_1mg5_unfixed_1_0
Backbone RMSD: 2.68 A Moderate designability

enzyme_design_M0255_1mg5_unfixed_1_1
Backbone RMSD: 1.61 A Good designability

enzyme_design_M025

In [14]:
from atomworks.io.utils.io_utils import to_cif_file

# Export structures to CIF format for visualization in PyMOL/ChimeraX
to_cif_file(aa_generated, "generated.cif")
to_cif_file(aa_refolded, "refolded.cif")

print("Exported structures:")
print("  - generated.cif: Original RFD3 backbone")
print("  - refolded.cif:  RF3-predicted structure")

Exported structures:
  - generated.cif: Original RFD3 backbone
  - refolded.cif:  RF3-predicted structure


### Superimposed Result

The image below shows the generated backbone (RFD3) superimposed with the re-folded structure (RF3). Close alignment indicates successful design.

![Superimposed Protein](../docs/_static/superimposed_80_residue_protein.png)